In [0]:
from pyspark.sql import functions as F

In [0]:
# Objetivo: personalização e segmentação de clientes
# LGPD: usa apenas o user_id (pseudonimizado) — sem nome, CPF ou e-mail

silver_users  = spark.table("silver.users")
fact_tickets  = spark.table("gold_bi.fact_ticket_sales")
silver_sales  = spark.table("silver.sales")

# RFM por usuário (Recency, Frequency, Monetary)
from pyspark.sql.window import Window

rfm_window = Window.partitionBy("user_id")

rfm = (
    fact_tickets
    .filter(F.col("order_status") == "confirmed")
    .withColumn("max_date", F.max("date_key").over(rfm_window))
    .groupBy("user_id")
    .agg(
        F.max("date_key").alias("last_purchase_date_key"),   # Recency
        F.count("order_id").alias("total_orders"),           # Frequency
        F.sum("unit_price_brl").alias("total_spent_brl"),    # Monetary
        F.countDistinct("event_id").alias("events_attended"),
        F.countDistinct("venue_city").alias("cities_visited"),
        F.avg("unit_price_brl").alias("avg_ticket_price"),
    )
)

(rfm.join(silver_users.select("id", "sex", "birth_year"),
          rfm.user_id == silver_users.id, "left")
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_ai.features_consumer_behavior"))